# Corrected thesis experiments — MD MAHMUDUL HASAN
## Run this notebook in Google Colab
This notebook preserves the original experiment organization and figure/table exports for the original thesis. New controls are exported as additional tables. Numerical results must come from a completed final run.

1. Keep your four original CSV files in `/content/drive/MyDrive/thesis_data`.
2. Select a GPU runtime if available. Run all cells. The default `QUICK_MODE=False` produces the full experiment; use True only for a smoke test.
3. Download the ZIP printed by the last cell and upload it to ChatGPT. Also save this notebook with outputs.
4. Reconnect and run all cells after a disconnect; configuration-isolated atomic checkpoints resume completed runs.

Changes: uniform priority sampling; explicit labels; feature-vector deduplication; common target tests; target-only and pooled controls; nested budgets; feature ablations; descriptive target-level statistics; complete logs and input hashes.

The five model seeds share a fixed partition. Their standard deviations measure model/adaptation variation, not independent networks. Optional extensions such as temporal evaluation and imputation sensitivity are not implemented in this version and must not be claimed in the thesis. Raw datasets are not included in the export.


## Part 0: Setup

In [ ]:
!pip install -q xgboost shap

In [ ]:
import os, json, time, glob, platform, warnings, zipfile, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
import sklearn
import torch
import torch.nn as nn
from scipy.stats import ks_2samp
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, precision_score, average_precision_score, confusion_matrix

# Preserve warnings for diagnosis.
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 10, 'font.family': 'serif'})
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
# ===================== CONFIGURATION =====================
QUICK_MODE = False          # True = fast test run. Set False for the FINAL thesis run.

# Google Drive folders (change if your folders are named differently)
DATA_DIR    = '/content/drive/MyDrive/thesis_data'      # put the 4 CSV files here
RESULTS_ROOT = '/content/drive/MyDrive/thesis_results_corrected'   # all tables and figures are saved here

# File name patterns for each dataset inside DATA_DIR (glob patterns)
DATASETS = {
    'ToN-IoT':      'NF-ToN-IoT-v3*.csv',
    'BoT-IoT':      'NF-BoT-IoT-v3*.csv',
    'UNSW-NB15':    'NF-UNSW-NB15-v3*.csv',
    'CIC-IDS2018':  '*CIC*2018*.csv',
}
IOT_DATASETS = ['ToN-IoT', 'BoT-IoT']    # used to mark IoT datasets in tables and plots

# Sampling caps per dataset (the full datasets have tens of millions of rows)
CAP_BENIGN       = 30_000 if QUICK_MODE else 150_000   # max benign flows per dataset
CAP_PER_ATTACK   = 5_000  if QUICK_MODE else 30_000    # max flows per original attack type per dataset
CHUNK_SIZE       = 1_000_000

SEEDS            = [0, 1] if QUICK_MODE else [0, 1, 2, 3, 4]   # thesis uses 5 seeds
TUNE_ITER        = 3 if QUICK_MODE else 10                      # random search iterations
NN_EPOCHS        = 5 if QUICK_MODE else 20
FEW_SHOT_FRACS   = [0.01, 0.05, 0.10]
MINIMAL_K        = 10      # number of features for strategy S2
TUNE_MAX_ROWS    = 50_000 if QUICK_MODE else 150_000   # tuning uses a subsample of the training data (speed)
MAX_TRAIN_ROWS   = 150_000 if QUICK_MODE else 800_000  # cap on training rows for multi-source training (Colab memory/time)

# Checkpointing: finished runs are saved to Drive, so if Colab disconnects
# you can simply run all cells again and finished work is skipped.

# Connect Google Drive FIRST (before any folder is created), then create the results folders
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
assert os.path.isdir(DATA_DIR), f'Folder not found: {DATA_DIR}. Create a folder named thesis_data in My Drive and put the CSV files in it.'
import hashlib, pathlib, inspect, sys
CODE_VERSION = 'corrected-v1.0'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

print('Fingerprinting input CSV files once; this may take several minutes.')
INPUTS = []
for dataset, pattern in DATASETS.items():
    files = sorted(glob.glob(f'{DATA_DIR}/**/{pattern}', recursive=True))
    assert files, f'Missing dataset: {dataset}'
    for path in files:
        INPUTS.append({'dataset':dataset, 'name':os.path.basename(path), 'bytes':os.path.getsize(path), 'sha256':sha256_file(path)})
CONFIG = dict(code=CODE_VERSION, quick=QUICK_MODE, benign_cap=CAP_BENIGN, attack_cap=CAP_PER_ATTACK,
              seeds=SEEDS, tune=TUNE_ITER, epochs=NN_EPOCHS, fractions=FEW_SHOT_FRACS,
              minimal_k=MINIMAL_K, tune_cap=TUNE_MAX_ROWS, train_cap=MAX_TRAIN_ROWS,
              sampling_seed=42, split_seed=123, inputs=INPUTS, device=DEVICE,
              versions={m.__name__:m.__version__ for m in [np,pd,sklearn,xgb,torch,shap]})
RUN_ID = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:16]
RESULTS_DIR = f'{RESULTS_ROOT}/{"quick" if QUICK_MODE else "final"}_{RUN_ID}'
for sub in ['tables','figures','logs','checkpoints']:
    os.makedirs(f'{RESULTS_DIR}/{sub}',exist_ok=True)
json.dump(CONFIG,open(f'{RESULTS_DIR}/logs/config.json','w'),indent=2)
print('Results:',RESULTS_DIR)
def save_table(df,name):
    df.to_csv(f'{RESULTS_DIR}/tables/{name}.csv',index=False)
def save_fig(fig,name):
    fig.savefig(f'{RESULTS_DIR}/figures/{name}.png',bbox_inches='tight')
def cache_load(exp):
    keys,rows=set(),[]
    for path in sorted(glob.glob(f'{RESULTS_DIR}/checkpoints/{exp}_*.json')):
        obj=json.load(open(path)); assert obj['run_id']==RUN_ID
        keys.add(obj['key']);rows.extend(obj['rows'])
    return keys,rows
def cache_add(exp,key,rows):
    path=f'{RESULTS_DIR}/checkpoints/{exp}_{hashlib.sha256(key.encode()).hexdigest()}.json'
    obj={'run_id':RUN_ID,'key':key,'rows':rows}
    with open(path+'.tmp','w') as f: json.dump(obj,f,default=lambda x:x.item() if hasattr(x,'item') else str(x))
    os.replace(path+'.tmp',path)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

In [ ]:
# Check that every dataset file is found
for name, pat in DATASETS.items():
    files = glob.glob(f'{DATA_DIR}/**/{pat}', recursive=True)
    print(f'{name:12s} ->', files if files else 'NOT FOUND (check DATA_DIR and file name)')

In [ ]:
# Record the environment for Table 4.1 of the thesis
env = {
    'Python': platform.python_version(), 'Platform': platform.platform(),
    'GPU': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
    'pandas': pd.__version__, 'numpy': np.__version__, 'scikit-learn': sklearn.__version__,
    'xgboost': xgb.__version__, 'torch': torch.__version__, 'shap': shap.__version__,
    'QUICK_MODE': QUICK_MODE, 'Seeds': str(SEEDS),
}
try:
    import psutil; env['RAM_GB'] = round(psutil.virtual_memory().total / 1e9, 1)
except Exception: pass
save_table(pd.DataFrame(list(env.items()), columns=['Item', 'Value']), 'table4_1_environment')
pd.DataFrame(list(env.items()), columns=['Item', 'Value'])

## Part 1: Load the datasets

**Datasets (NetFlow v3, University of Queensland):** NF-ToN-IoT-v3, NF-BoT-IoT-v3, NF-UNSW-NB15-v3, NF-CSE-CIC-IDS2018-v3.
Download page: https://staff.itee.uq.edu.au/marius/NIDS_datasets/

All four share the same 53 NetFlow features, so their feature definitions are identical. This makes the cross-dataset comparison fair (thesis Section 3.5).
Unzip each download and upload the CSV into `DATA_DIR`.

The full files are very large, so we read them in chunks and keep a **stratified sample**: up to `CAP_BENIGN` benign flows and up to `CAP_PER_ATTACK` flows of each attack type. Sampling is random (fixed seed) and happens before any train/test split.

In [ ]:
LABEL_COL, ATTACK_COL = 'Label', 'Attack'
def load_sampled(pattern, seed=42):
    files=sorted(glob.glob(f'{DATA_DIR}/**/{pattern}',recursive=True))
    assert files
    rng=np.random.default_rng(seed); kept={}; counts_total={}
    for f in files:
        offset=0
        for chunk in pd.read_csv(f,chunksize=CHUNK_SIZE,low_memory=False):
            chunk[ATTACK_COL]=chunk[ATTACK_COL].astype(str).str.strip()
            chunk['_priority']=rng.random(len(chunk))
            chunk['_row_id']=[f'{os.path.basename(f)}:{j}' for j in range(offset,offset+len(chunk))]
            offset+=len(chunk)
            for cls,g in chunk.groupby(ATTACK_COL):
                counts_total[cls]=counts_total.get(cls,0)+len(g)
                cap=CAP_BENIGN if cls.lower()=='benign' else CAP_PER_ATTACK
                prev=kept.get(cls)
                combined=pd.concat([prev,g],ignore_index=True) if prev is not None else g
                kept[cls]=combined.nsmallest(cap,'_priority')
    return pd.concat(kept.values(),ignore_index=True).drop(columns='_priority'),counts_total
raw,full_counts={},{}
for name,pat in DATASETS.items():
    raw[name],full_counts[name]=load_sampled(pat)
    print(name,len(raw[name]),'sampled rows')


## Part 2: Cleaning, identifier removal, label harmonization

* **Identifier features are removed** (IP addresses, ports, flow start/end timestamps, DNS query ID). They identify hosts and times in one testbed, so a model could memorize them instead of learning attack behavior (Arp et al., 2022).
* Only features present in **all** datasets are kept.
* Missing and infinite values are removed, then **exact duplicates are removed** so the same record cannot appear in both train and test.
* Original attack names are mapped to shared classes. **Check the printed mapping table carefully** and correct `harmonize_label` if anything looks wrong.

In [ ]:
IDENTIFIERS = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT',
               'FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'DNS_QUERY_ID']

LABEL_MAP = {
 'benign':'Benign','injection':'Web/Injection','xss':'Web/Injection',
 'password':'Brute force','dos':'DoS/DDoS','ddos':'DoS/DDoS',
 'backdoor':'Botnet/Malware','mitm':'Other','scanning':'Reconnaissance',
 'ransomware':'Botnet/Malware','reconnaissance':'Reconnaissance','theft':'Other',
 'exploits':'Other','fuzzers':'Other','generic':'Other','shellcode':'Botnet/Malware',
 'analysis':'Other','worms':'Botnet/Malware','dos_attacks-hulk':'DoS/DDoS',
 'ddos_attacks-loic-http':'DoS/DDoS','ssh-bruteforce':'Brute force',
 'dos_attacks-goldeneye':'DoS/DDoS','ddos_attack-hoic':'DoS/DDoS',
 'infilteration':'Botnet/Malware','infiltration':'Botnet/Malware',
 'dos_attacks-slowloris':'DoS/DDoS','ddos_attack-loic-udp':'DoS/DDoS',
 'bot':'Botnet/Malware','brute_force_-web':'Brute force',
 'sql_injection':'Web/Injection','brute_force_-xss':'Web/Injection',
 'dos_attacks-slowhttptest':'DoS/DDoS','ftp-bruteforce':'Brute force'}
def harmonize_label(a):
    key=str(a).strip().lower()
    if key not in LABEL_MAP: raise ValueError(f'Unknown attack label: {a!r}; review mapping before continuing.')
    return LABEL_MAP[key]

common = set.intersection(*[set(df.columns) for df in raw.values()])
FEATURES = sorted(c for c in common if c not in IDENTIFIERS + [LABEL_COL, ATTACK_COL, '_row_id'])
print(f'{len(FEATURES)} shared features used:\n', FEATURES)

data, stats_rows, map_rows = {}, [], []
for name, df in raw.items():
    n0 = len(df)
    df = df[FEATURES + [LABEL_COL, ATTACK_COL, '_row_id']].copy()
    df[FEATURES] = df[FEATURES].apply(pd.to_numeric, errors='coerce')
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    n1 = len(df)
    df['y'] = (pd.to_numeric(df[LABEL_COL],errors='raise') > 0).astype(int)
    df['cls'] = df[ATTACK_COL].map(harmonize_label)
    hashes=pd.util.hash_pandas_object(df[FEATURES],index=False)
    df['_feature_hash']=hashes.astype(str)
    conflict=df.groupby('_feature_hash').y.transform('nunique').gt(1)
    ambiguous=df.groupby('_feature_hash').cls.transform('nunique').gt(1)
    df.loc[ambiguous,'cls']='Ambiguous'
    df=df[~conflict].sort_values('_row_id').drop_duplicates(subset=FEATURES).copy()
    assert not df.duplicated(subset=FEATURES).any()
    df['dataset'] = name
    data[name] = df.reset_index(drop=True)
    for a, c in df[ATTACK_COL].value_counts().items():
        map_rows.append({'Dataset': name, 'Original label': a, 'Harmonized class': harmonize_label(a),
                         'Flows in full dataset': full_counts[name].get(a, ''), 'Flows after sampling and cleaning': c})
    stats_rows.append({'Dataset': name, 'IoT': name in IOT_DATASETS,
                       'Flows in full dataset': sum(full_counts[name].values()),
                       'Sampled': n0, 'After removing invalid': n1, 'After deduplication': len(df),
                       'Benign': int((df.y == 0).sum()), 'Attack': int((df.y == 1).sum()),
                       'Attack %': round(100 * df.y.mean(), 2), 'Features': len(FEATURES)})

stats = pd.DataFrame(stats_rows); save_table(stats, 'table4_2_dataset_statistics')
mapping = pd.DataFrame(map_rows); save_table(mapping, 'table3_2_label_mapping')
pd.set_option('display.max_rows', 200)
display(stats); display(mapping)
# Save sample provenance and exact-feature overlap counts, without raw network identifiers.
for name,df in data.items():
    save_table(df[['_row_id','_feature_hash','y','cls']],f'sample_manifest_{name}')
overlap=[]
for a in data:
    for b in data:
        if a!=b: overlap.append({'Source':a,'Target':b,'Shared feature hashes':len(set(data[a]._feature_hash)&set(data[b]._feature_hash))})
save_table(pd.DataFrame(overlap),'cross_dataset_overlap_audit')
assert len(FEATURES)==46, f'Unexpected schema: {len(FEATURES)} features. Inspect before proceeding.'


In [ ]:
# Figure: class distribution per dataset (harmonized classes)
dist = pd.concat(data.values()).groupby(['dataset', 'cls']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(8, 4))
(dist.div(dist.sum(1), axis=0) * 100).plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_ylabel('Share of flows (%)'); ax.set_xlabel(''); ax.set_title('Harmonized class distribution after sampling')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8); plt.xticks(rotation=0)
save_fig(fig, 'fig4_1_class_distribution'); plt.show()

## Shared functions: preprocessing, models, metrics

* Features are log-transformed, `sign(x) * log(1 + |x|)`, because flow counts and byte counts are extremely skewed. This transform has no fitted parameters, so it cannot leak test information.
* The standard scaler for neural networks is **fitted on training data only**.
* Class imbalance is handled with class weights computed from the training data only.
* Hyperparameters are tuned by random search on a **validation set taken from the training data only**. The test or target data is never used for tuning.

In [ ]:
def prep_X(df, feats=None):
    X = df[feats or FEATURES].values.astype(np.float64)
    return np.sign(X) * np.log1p(np.abs(X))

def metrics(y_true, score, thr=0.5):
    y_true = np.asarray(y_true); pred = (score >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {'Macro F1': f1_score(y_true, pred, average='macro', zero_division=0),
            'Precision': precision_score(y_true, pred, zero_division=0),
            'Recall': recall_score(y_true, pred, zero_division=0),
            'FPR': fp / (fp + tn) if (fp + tn) else np.nan,
            'PR-AUC': average_precision_score(y_true, score) if len(np.unique(y_true)) > 1 else np.nan,
            'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn}

# ---------- Random Forest ----------
RF_SPACE = {'n_estimators': [100, 200], 'max_depth': [10, 20, None],
            'min_samples_leaf': [1, 2, 5], 'max_features': ['sqrt', 0.5]}
def fit_rf(X, y, p, seed):
    m = RandomForestClassifier(**p, class_weight='balanced', n_jobs=-1, random_state=seed)
    return m.fit(X, y)
def score_rf(m, X): return m.predict_proba(X)[:, 1]

# ---------- XGBoost ----------
XGB_SPACE = {'n_estimators': [200, 400], 'max_depth': [4, 6, 8], 'learning_rate': [0.05, 0.1, 0.2],
             'subsample': [0.8, 1.0], 'colsample_bytree': [0.7, 1.0]}
def fit_xgb(X, y, p, seed):
    spw = (y == 0).sum() / max((y == 1).sum(), 1)
    m = xgb.XGBClassifier(**p, tree_method='hist', device=DEVICE, scale_pos_weight=spw,
                          eval_metric='logloss', random_state=seed, n_jobs=-1)
    return m.fit(X, y)
def score_xgb(m, X): return m.predict_proba(X)[:, 1]

# ---------- MLP (PyTorch; also the base network for DANN) ----------
MLP_SPACE = {'hidden': [(128, 64), (256, 128), (64, 32)], 'lr': [1e-3, 3e-4], 'dropout': [0.1, 0.3]}

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam): ctx.lam = lam; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lam * g, None

class Net(nn.Module):
    def __init__(self, d, hidden=(128, 64), dropout=0.1):
        super().__init__()
        layers, prev = [], d
        for h in hidden: layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]; prev = h
        self.feat = nn.Sequential(*layers)
        self.clf = nn.Linear(prev, 1)
        self.dom = nn.Sequential(nn.Linear(prev, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x, lam=0.0, domain=False):
        z = self.feat(x)
        if domain: return self.clf(z).squeeze(1), self.dom(GradReverse.apply(z, lam)).squeeze(1)
        return self.clf(z).squeeze(1)

class NNModel:
    """Wrapper so the neural network behaves like the other models."""
    def __init__(self, net, scaler): self.net, self.scaler = net, scaler

def _batches(n, bs, rng):
    idx = rng.permutation(n)
    for i in range(0, n, bs): yield idx[i:i + bs]

def fit_mlp(X, y, p, seed, Xt_unlab=None, epochs=None, init=None):
    """Plain MLP when Xt_unlab is None; DANN when unlabeled target data is given."""
    set_seed(seed); rng = np.random.default_rng(seed); epochs = epochs or NN_EPOCHS
    if init is None:
        scaler = StandardScaler().fit(X)
        net = Net(X.shape[1], p['hidden'], p['dropout']).to(DEVICE)
    else:
        scaler, net = init.scaler, init.net
    Xs = torch.tensor(scaler.transform(X), dtype=torch.float32, device=DEVICE)
    ys = torch.tensor(y, dtype=torch.float32, device=DEVICE)
    pos_w = torch.tensor((y == 0).sum() / max((y == 1).sum(), 1), dtype=torch.float32, device=DEVICE)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pos_w); dlossf = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(net.parameters(), lr=p['lr'])
    if Xt_unlab is not None:
        Xt = torch.tensor(scaler.transform(Xt_unlab), dtype=torch.float32, device=DEVICE)
    bs = 1024; steps = epochs * int(np.ceil(len(X) / bs)); step = 0
    for ep in range(epochs):
        net.train()
        for b in _batches(len(X), bs, rng):
            if len(b) < 2: continue
            if Xt_unlab is None:
                loss = lossf(net(Xs[b]), ys[b])
            else:
                prog = step / steps; lam = 2 / (1 + np.exp(-10 * prog)) - 1   # DANN schedule (Ganin et al., 2016)
                tb = torch.tensor(rng.integers(0, len(Xt), len(b)), device=DEVICE)
                xin = torch.cat([Xs[b], Xt[tb]])
                cl, dl = net(xin, lam, domain=True)
                dlab = torch.cat([torch.zeros(len(b)), torch.ones(len(b))]).to(DEVICE)
                loss = lossf(cl[:len(b)], ys[b]) + dlossf(dl, dlab)
            opt.zero_grad(); loss.backward(); opt.step(); step += 1
    return NNModel(net, scaler)

def score_mlp(m, X):
    m.net.eval(); out = []
    Xs = torch.tensor(m.scaler.transform(X), dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        for i in range(0, len(Xs), 8192): out.append(torch.sigmoid(m.net(Xs[i:i + 8192])).cpu().numpy())
    return np.concatenate(out)

MODELS = {'RF':      (RF_SPACE,  fit_rf,  score_rf),
          'XGBoost': (XGB_SPACE, fit_xgb, score_xgb),
          'MLP':     (MLP_SPACE, fit_mlp, score_mlp)}

def sample_params(space, rng):
    return {k: v[rng.integers(len(v))] for k, v in space.items()}

def tune(model_name, Xtr, ytr, Xval, yval, seed=0):
    """Random search; selection uses validation data from the training side only."""
    space, fit, score = MODELS[model_name]; rng = np.random.default_rng(seed)
    best, best_f1, tried = None, -1, set()
    for _ in range(TUNE_ITER * 20):
        p = sample_params(space, rng); key = json.dumps(p, sort_keys=True, default=str)
        if key in tried: continue
        tried.add(key)
        f1 = metrics(yval, score(fit(Xtr, ytr, p, seed), Xval))['Macro F1']
        if f1 > best_f1: best, best_f1 = p, f1
        if len(tried) >= TUNE_ITER: break
    return best

TUNED = {}  # Recomputed deterministically; parameters are archived with each run.

## Splits
* **In-dataset:** each dataset is split 35% train, 15% validation, 50% common test (stratified).
* **Target side:** each dataset is also split 50/50 into an *adaptation pool* and a *target test set*. All cross-dataset results are measured on the same target test set, so every strategy is compared on identical data. The adaptation pool supplies unlabeled data for DANN (S3) and the small labeled samples for fine-tuning (S4).

In [ ]:
SPLITS={}
for name,df in data.items():
    idx=np.arange(len(df))
    adapt,ttest=train_test_split(idx,test_size=.5,stratify=df.y,random_state=123)
    tr,va=train_test_split(adapt,test_size=.3,stratify=df.y.iloc[adapt],random_state=42)
    SPLITS[name]={'train':tr,'val':va,'test':ttest,'adapt':adapt,'ttest':ttest}
    assert not set(adapt)&set(ttest)
    ts,trs=set(ttest),set(tr)
    save_table(pd.DataFrame({'row':idx,'feature_hash':df._feature_hash,
        'split':['test' if i in ts else 'train' if i in trs else 'val' for i in idx]}),f'splits_{name}')
def part(name,split):return data[name].iloc[SPLITS[name][split]]
def strat_sample(df,n,seed):
    # Nested prefixes of fixed per-class permutations. Exact count may differ by one due to rounding.
    rng=np.random.default_rng(seed);keep=[]
    for _,idx in df.groupby('y',sort=True).indices.items():
        k=max(1,min(len(idx),int(round(n*len(idx)/len(df)))))
        keep.extend(rng.permutation(idx)[:k])
    return df.iloc[sorted(keep)]
def cap_rows(df,n,seed):return df if len(df)<=n else strat_sample(df,n,seed)


## Part 3: Experiment 1, in-dataset baseline (Thesis Section 5.1)
Each model is tuned and trained on one dataset and tested on the same dataset. The same trained models are also tested on the **other three** datasets, which gives the single-source transfer matrix used in Part 4.

In [ ]:
keys, rows_e1 = cache_load('exp1')
for src in data:
    tr, va = part(src, 'train'), part(src, 'val')
    Xtr, ytr = prep_X(tr), tr.y.values
    Xva, yva = prep_X(va), va.y.values
    tr_s = cap_rows(tr, TUNE_MAX_ROWS, 0)
    for mname, (space, fit, score) in MODELS.items():
        pkey = f'{src}|{mname}'
        if pkey not in TUNED:
            TUNED[pkey] = tune(mname, prep_X(tr_s), tr_s.y.values, Xva, yva)
        for seed in SEEDS:
            key = f'{src}|{mname}|{seed}'
            if key in keys: continue
            t0 = time.time(); m = fit(Xtr, ytr, TUNED[pkey], seed); ttrain = time.time() - t0
            new = []
            for tgt in data:
                ev = part(src, 'test') if tgt == src else part(tgt, 'ttest')
                t0 = time.time(); sc = score(m, prep_X(ev)); tinf = time.time() - t0
                r = {'Source': src, 'Target': tgt, 'Model': mname, 'Seed': seed, **metrics(ev.y.values, sc),
                     'Train time (s)': ttrain, 'Inference time per 1k flows (ms)': 1000 * tinf / len(ev) * 1000}
                # per-class detection rate on the target (used for RQ1)
                for c, g in pd.DataFrame({'cls': ev.cls.values, 'p': sc >= 0.5}).groupby('cls'):
                    r[f'DR {c}'] = g.p.mean()
                new.append(r)
            cache_add('exp1', key, new); rows_e1 += new
            print(f'{key}: in-dataset macro F1 = {new[list(data).index(src)]["Macro F1"]:.4f}  ({ttrain:.0f}s)')

E1 = pd.DataFrame(rows_e1)
json.dump(TUNED, open(f'{RESULTS_DIR}/logs/tuned_params.json', 'w'), indent=1, default=str)

def mean_std(df, by, cols):
    g = df.groupby(by)[cols]
    out = g.mean().round(4).astype(str) + ' ± ' + g.std().fillna(0).round(4).astype(str)
    return out.reset_index()

IN = E1[E1.Source == E1.Target]
t51 = mean_std(IN, ['Source', 'Model'], ['Macro F1', 'Precision', 'Recall', 'FPR', 'PR-AUC']).rename(columns={'Source': 'Dataset'})
t51 = t51.merge(IN.groupby(['Source', 'Model'])[['Train time (s)', 'Inference time per 1k flows (ms)']].mean().round(2).reset_index().rename(columns={'Source': 'Dataset'}))
save_table(t51, 'table5_1_in_dataset'); display(t51)

## Part 4: Experiment 2, cross-dataset evaluation (RQ1, Thesis Section 5.2)
**4a. Single-source transfer matrix:** rows = training dataset, columns = test dataset. The diagonal is the in-dataset result.

In [ ]:
pair = E1.groupby(['Model', 'Source', 'Target'])['Macro F1'].mean().reset_index()
order = list(data)
fig, axes = plt.subplots(1, len(MODELS), figsize=(5.5 * len(MODELS), 4.4), sharey=True)
for i, (ax, mname) in enumerate(zip(np.atleast_1d(axes), MODELS)):
    M = pair[pair.Model == mname].pivot(index='Source', columns='Target', values='Macro F1').loc[order, order]
    sns.heatmap(M, annot=True, fmt='.2f', vmin=0, vmax=1, cmap='RdYlGn', ax=ax, cbar=i == len(MODELS) - 1)
    ax.set_title(mname); ax.set_xlabel('Test dataset'); ax.set_ylabel('Training dataset' if i == 0 else '')
fig.tight_layout()
fig.suptitle('Macro F1 for every training/test dataset pair (diagonal = in-dataset)', y=1.02)
save_fig(fig, 'fig5_1_transfer_matrix'); plt.show()
save_table(pair.pivot_table(index=['Model', 'Source'], columns='Target', values='Macro F1').round(4).reset_index(), 'table5_2a_transfer_matrix')

**4b. Leave-one-dataset-out (LODO), which is also strategy S1 (multi-source training):** train on the other three datasets, test on the held-out dataset. Hyperparameters are tuned on validation data from the three source datasets only.

In [ ]:
def lodo_sources(tgt): return [d for d in data if d != tgt]
def source_union(tgt, split, seed=0, cap=None):
    df = pd.concat([part(s, split) for s in lodo_sources(tgt)])
    return cap_rows(df, cap, seed) if cap else df

keys, rows_e2 = cache_load('exp2')
LODO_MODELS = {}   # keep XGBoost and MLP models of seed 0 for later strategies and SHAP
for tgt in data:
    trn = source_union(tgt, 'train', cap=MAX_TRAIN_ROWS); val = source_union(tgt, 'val', cap=TUNE_MAX_ROWS)
    Xtr, ytr, Xva, yva = prep_X(trn), trn.y.values, prep_X(val), val.y.values
    ev = part(tgt, 'ttest'); Xev = prep_X(ev)
    tr_s = cap_rows(trn, TUNE_MAX_ROWS, 0)
    for mname, (space, fit, score) in MODELS.items():
        pkey = f'LODO-{tgt}|{mname}'
        if pkey not in TUNED: TUNED[pkey] = tune(mname, prep_X(tr_s), tr_s.y.values, Xva, yva)
        for seed in SEEDS:
            key = f'{tgt}|{mname}|{seed}'
            if key in keys and not (seed == SEEDS[0] and mname != 'RF'): continue
            t0 = time.time(); m = fit(Xtr, ytr, TUNED[pkey], seed); ttrain = time.time() - t0
            if seed == SEEDS[0] and mname != 'RF': LODO_MODELS[(tgt, mname)] = m
            if key in keys: continue
            sc = score(m, Xev)
            r = {'Target': tgt, 'Model': mname, 'Seed': seed, **metrics(ev.y.values, sc), 'Train time (s)': ttrain}
            for c, g in pd.DataFrame({'cls': ev.cls.values, 'p': sc >= 0.5}).groupby('cls'): r[f'DR {c}'] = g.p.mean()
            cache_add('exp2', key, [r]); rows_e2.append(r)
            print(f'LODO target={tgt} {mname} seed={seed}: macro F1 = {r["Macro F1"]:.4f}')
json.dump(TUNED, open(f'{RESULTS_DIR}/logs/tuned_params.json', 'w'), indent=1, default=str)
E2 = pd.DataFrame(rows_e2)

In [ ]:
# Table 5.2: generalization gap = in-dataset F1 minus LODO F1 on the same target
ind = IN.groupby(['Source', 'Model'])['Macro F1'].mean().rename('In-dataset F1').reset_index().rename(columns={'Source': 'Target'})
lod = E2.groupby(['Target', 'Model']).agg(**{'LODO F1': ('Macro F1', 'mean'), 'LODO F1 std': ('Macro F1', 'std'),
                                            'LODO FPR': ('FPR', 'mean'), 'LODO Recall': ('Recall', 'mean')}).reset_index()
fpr_in = IN.groupby(['Source', 'Model'])['FPR'].mean().rename('In-dataset FPR').reset_index().rename(columns={'Source': 'Target'})
t52 = ind.merge(lod).merge(fpr_in)
t52['Gap'] = t52['In-dataset F1'] - t52['LODO F1']
t52['IoT target'] = t52.Target.isin(IOT_DATASETS)
t52 = t52.round(4); save_table(t52, 'table5_2_generalization_gap'); display(t52)

In [ ]:
# Figure 5.2: in-dataset vs LODO macro F1
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
long = t52.melt(id_vars=['Target', 'Model'], value_vars=['In-dataset F1', 'LODO F1'], var_name='Setting', value_name='Macro F1')
sns.barplot(data=long[long.Setting == 'In-dataset F1'], x='Target', y='Macro F1', hue='Model', ax=axes[0])
axes[0].set_title('In-dataset'); axes[0].set_ylim(0, 1.05)
sns.barplot(data=long[long.Setting == 'LODO F1'], x='Target', y='Macro F1', hue='Model', ax=axes[1])
axes[1].set_title('Leave-one-dataset-out (unseen target)'); axes[1].set_ylim(0, 1.05)
for ax in axes: ax.set_xlabel('Target dataset')
save_fig(fig, 'fig5_2_in_vs_lodo'); plt.show()

# Figure 5.3: false positive rate in-dataset vs LODO
fpr_long = t52.melt(id_vars=['Target', 'Model'], value_vars=['In-dataset FPR', 'LODO FPR'], var_name='Setting', value_name='FPR')
g = sns.catplot(data=fpr_long, x='Target', y='FPR', hue='Setting', col='Model', kind='bar', height=3.6, aspect=1.1)
g.set_axis_labels('Target dataset', 'False positive rate')
save_fig(g.fig, 'fig5_3_fpr'); plt.show()

In [ ]:
# Figure 5.4 and Table 5.3: detection rate per harmonized attack class on the unseen target (LODO)
dr_cols = [c for c in E2.columns if c.startswith('DR ') and c != 'DR Benign']
dr = E2.groupby(['Target', 'Model'])[dr_cols].mean()
dr.columns = [c[3:] for c in dr.columns]
save_table(dr.round(4).reset_index(), 'table5_3_detection_rate_per_class')
fig, axes = plt.subplots(1, len(MODELS), figsize=(5.5 * len(MODELS), 4), sharey=True)
for i, (ax, mname) in enumerate(zip(np.atleast_1d(axes), MODELS)):
    sns.heatmap(dr.xs(mname, level='Model').loc[order], annot=True, fmt='.2f', vmin=0, vmax=1, cmap='RdYlGn', ax=ax,
                cbar=i == len(MODELS) - 1)
    ax.set_title(f'{mname}'); ax.set_xlabel('Attack class'); ax.set_ylabel('Target dataset' if i == 0 else '')
fig.suptitle('Attack detection rate on the unseen target dataset (blank = class not present)', y=1.02); fig.tight_layout()
save_fig(fig, 'fig5_4_detection_rate_per_class'); plt.show()
display(dr.round(3))

## Part 5: Experiment 3, generalization strategies (RQ2, Thesis Section 5.3)
All strategies are evaluated on the same target test set.

| Strategy | Idea | Needs from target |
|---|---|---|
| Single-source (baseline) | Train on one dataset, average over the three possible sources | nothing |
| **S1** Multi-source | Train on all three source datasets (= LODO in Part 4) | nothing |
| **S2** Minimal stable features | Keep only `MINIMAL_K` features that are important **and** have similar distributions across the source datasets (Kolmogorov-Smirnov statistic) | nothing |
| **S3** DANN | Domain-adversarial training (Ganin et al., 2016) with unlabeled target flows (MLP only) | unlabeled flows |
| **S4** Few-shot fine-tuning | Continue training the S1 model on 1%, 5%, 10% labeled target flows | few labeled flows |

In [ ]:
def ks_shift(dfs, feats, n=20_000, seed=0):
    """Mean pairwise KS statistic per feature across a list of dataframes."""
    samp = [prep_X(d.sample(min(n, len(d)), random_state=seed), feats) for d in dfs]
    out = {}
    for j, f in enumerate(feats):
        vals = [ks_2samp(samp[a][:, j], samp[b][:, j]).statistic for a in range(len(samp)) for b in range(a + 1, len(samp))]
        out[f] = float(np.mean(vals))
    return pd.Series(out)

def stable_features(tgt, k=MINIMAL_K):
    srcs = [part(s, 'train') for s in lodo_sources(tgt)]
    shift = ks_shift(srcs, FEATURES)
    imp = pd.Series(LODO_MODELS[(tgt, 'XGBoost')].feature_importances_, index=FEATURES)
    cand = shift[shift <= shift.median()].index
    return list(imp[cand].sort_values(ascending=False).index[:k]), shift

import copy
keys, rows_e3 = cache_load('exp3')
S2_FEATS = {}
for tgt in data:
    trn = source_union(tgt, 'train', cap=MAX_TRAIN_ROWS); ev = part(tgt, 'ttest'); adapt = part(tgt, 'adapt')
    S2_FEATS[tgt], _ = stable_features(tgt)
    for mname in ['XGBoost', 'MLP']:
        space, fit, score = MODELS[mname]; p = TUNED[f'LODO-{tgt}|{mname}']
        for seed in SEEDS:
            def run(strategy, frac, fn):
                key = f'{tgt}|{mname}|{seed}|{strategy}|{frac}'
                if key in keys: return
                sc = fn()
                r = {'Target': tgt, 'Model': mname, 'Seed': seed, 'Strategy': strategy, 'Target labels used (%)': 100 * frac,
                     'Label budget denominator':len(data[tgt]), 'Approx target labels':int(frac*len(data[tgt])), **metrics(ev.y.values, sc)}
                cache_add('exp3', key, [r]); rows_e3.append(r)
                print(f'{tgt:12s} {mname:8s} seed={seed} {strategy:28s} F1={r["Macro F1"]:.4f}')
            # S2: minimal stable features
            f2 = S2_FEATS[tgt]
            run('S2 Minimal stable features', 0, lambda: score(fit(prep_X(trn, f2), trn.y.values, p, seed), prep_X(ev, f2)))
            # S3: DANN (neural network only)
            if mname == 'MLP':
                run('S3 DANN', 0, lambda: score_mlp(fit_mlp(prep_X(trn), trn.y.values, p, seed,
                                                    Xt_unlab=prep_X(cap_rows(adapt, MAX_TRAIN_ROWS, seed))), prep_X(ev)))
            # S4: few-shot fine-tuning from the S1 (LODO) model
            if all(f'{tgt}|{mname}|{seed}|{st}|{f}' in keys for f in FEW_SHOT_FRACS for st in ['S4 Few-shot fine-tuning','T0 Target-only','P0 Pooled source and target']): continue
            base = fit(prep_X(trn), trn.y.values, p, seed)   # the S1 model for this seed
            for frac in FEW_SHOT_FRACS:
                n = max(int(frac * len(data[tgt])), 20)
                shot = strat_sample(adapt, n, seed)
                Xs_, ys_ = prep_X(shot), shot.y.values
                if mname == 'XGBoost':
                    def ft():
                        spw = (ys_ == 0).sum() / max((ys_ == 1).sum(), 1)
                        m2 = xgb.XGBClassifier(**{**p, 'n_estimators': 100}, tree_method='hist', device=DEVICE,
                                               scale_pos_weight=spw, eval_metric='logloss', random_state=seed)
                        m2.fit(Xs_, ys_, xgb_model=base.get_booster())
                        return m2.predict_proba(prep_X(ev))[:, 1]
                else:
                    def ft():
                        m2 = NNModel(copy.deepcopy(base.net), base.scaler)
                        m2 = fit_mlp(Xs_, ys_, {**p, 'lr': p['lr'] / 5}, seed, epochs=max(5, NN_EPOCHS // 2), init=m2)
                        return score_mlp(m2, prep_X(ev))
                run('S4 Few-shot fine-tuning', frac, ft)
                run('T0 Target-only', frac, lambda: score(fit(Xs_,ys_,p,seed),prep_X(ev)))
                pooled=pd.concat([trn,shot],ignore_index=True)
                run('P0 Pooled source and target', frac, lambda: score(fit(prep_X(pooled),pooled.y.values,p,seed),prep_X(ev)))
E3 = pd.DataFrame(rows_e3)
json.dump({k: v for k, v in S2_FEATS.items()}, open(f'{RESULTS_DIR}/logs/s2_features.json', 'w'), indent=1)

In [ ]:
from scipy.stats import wilcoxon
single = E1[(E1.Source != E1.Target) & E1.Model.isin(['XGBoost', 'MLP'])].groupby(['Target', 'Model', 'Seed'])[['Macro F1', 'FPR', 'Recall']].mean().reset_index()
single['Strategy'] = 'Single-source (average)'; single['Target labels used (%)'] = 0
s1 = E2[E2.Model.isin(['XGBoost', 'MLP'])][['Target', 'Model', 'Seed', 'Macro F1', 'FPR', 'Recall']].copy()
s1['Strategy'] = 'S1 Multi-source'; s1['Target labels used (%)'] = 0
ALL = pd.concat([single, s1, E3[['Target', 'Model', 'Seed', 'Strategy', 'Target labels used (%)', 'Macro F1', 'FPR', 'Recall']]], ignore_index=True)
ALL['Label'] = ALL.apply(lambda r: r.Strategy if r.Strategy != 'S4 Few-shot fine-tuning' else f'S4 Fine-tune {r["Target labels used (%)"]:.0f}%', axis=1)
order_s = ['Single-source (average)', 'S1 Multi-source', 'S2 Minimal stable features', 'S3 DANN'] + [f'S4 Fine-tune {100*f:.0f}%' for f in FEW_SHOT_FRACS]
ALL.to_csv(f'{RESULTS_DIR}/tables/all_strategy_runs.csv', index=False)

t54 = ALL.groupby(['Model', 'Target', 'Label'])['Macro F1'].mean().unstack('Label').reindex(columns=[o for o in order_s if o in ALL.Label.unique()]).round(4)
parts = []
for mname in ['XGBoost', 'MLP']:
    blk = t54.loc[[mname]]
    mean_row = blk.mean().round(4).to_frame().T; mean_row.index = pd.MultiIndex.from_tuples([(mname, 'Mean over targets')])
    parts += [blk, mean_row]
t54 = pd.concat(parts); t54.index.names = ['Model', 'Target']
save_table(t54.reset_index(), 'table5_4_strategies_macro_f1'); display(t54)

# Conditional run variability; targets are not replicated by counting model seeds.
contrast=[]
for model in ['XGBoost','MLP']:
    for target in data:
        for frac in FEW_SHOT_FRACS:
            q=E3[(E3.Model==model)&(E3.Target==target)&np.isclose(E3['Target labels used (%)'],100*frac)]
            w=q.pivot(index='Seed',columns='Strategy',values='Macro F1')
            for strategy in ['S4 Few-shot fine-tuning','P0 Pooled source and target']:
                delta=w[strategy]-w['T0 Target-only']
                contrast.append({'Model':model,'Target':target,'Budget %':100*frac,'Strategy':strategy,
                    'Mean paired delta vs target-only':delta.mean(),'SD across seeds':delta.std(),'Runs':len(delta)})
save_table(pd.DataFrame(contrast),'table5_5_target_only_contrasts')
display(pd.DataFrame(contrast))


In [ ]:
# Figure 5.5: strategy comparison
plot = ALL[ALL.Label.isin(order_s)]
g = sns.catplot(data=plot, x='Target', y='Macro F1', hue='Label', hue_order=[o for o in order_s if o in plot.Label.unique()],
                col='Model', kind='bar', height=4, aspect=1.4, errorbar='sd', palette='viridis')
g.set_axis_labels('Unseen target dataset', 'Macro F1'); g.set(ylim=(0, 1.05)); g._legend.set_title('Strategy')
save_fig(g.fig, 'fig5_5_strategies'); plt.show()

# Figure 5.6: how much labeled target data is needed
curve = ALL[ALL.Label.isin(['S1 Multi-source']) | ALL.Strategy.eq('S4 Few-shot fine-tuning')]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, mname in zip(axes, ['XGBoost', 'MLP']):
    sns.lineplot(data=curve[curve.Model == mname], x='Target labels used (%)', y='Macro F1', hue='Target', marker='o', ax=ax, errorbar='sd')
    ax.set_title(mname); ax.set_xlabel('Labeled target flows used (% of target dataset)'); ax.set_ylim(0, 1.05)
save_fig(fig, 'fig5_6_few_shot_curve'); plt.show()

## Part 6: Experiment 4, explainability with SHAP (RQ3, Thesis Section 5.4)
1. Feature distribution shift between every pair of datasets (KS statistic).
2. SHAP importance of XGBoost single-source models for the **best** and **worst** transfer pair.
3. Does transfer fail more when the features the model relies on have shifted? We compute, for every source/target pair, the SHAP-weighted mean KS shift and correlate it with the transfer F1.

In [ ]:
from scipy.stats import spearmanr
names = list(data)
shift_pairs = {}
for a in range(len(names)):
    for b in range(a + 1, len(names)):
        shift_pairs[(names[a], names[b])] = shift_pairs[(names[b], names[a])] = ks_shift([data[names[a]], data[names[b]]], FEATURES)
mean_shift = pd.DataFrame({f'{a} vs {b}': s for (a, b), s in shift_pairs.items() if names.index(a) < names.index(b)})
mean_shift['Mean KS'] = mean_shift.mean(1)
mean_shift = mean_shift.sort_values('Mean KS', ascending=False)
save_table(mean_shift.round(4).reset_index().rename(columns={'index': 'Feature'}), 'table5_6_feature_shift_ks')

fig, ax = plt.subplots(figsize=(7, 8))
sns.heatmap(mean_shift.drop(columns='Mean KS'), cmap='magma_r', vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'KS statistic'})
ax.set_title('Feature distribution shift between datasets'); ax.set_xlabel('Dataset pair'); ax.set_ylabel('')
save_fig(fig, 'fig5_7_feature_shift'); plt.show()

In [ ]:
# Train one XGBoost per source (seed 0, tuned params) and compute SHAP on each target
xp = pair[pair.Model == 'XGBoost']
off = xp[xp.Source != xp.Target].sort_values('Macro F1')
worst, best = off.iloc[0], off.iloc[-1]
print('Worst transfer:', worst.Source, '->', worst.Target, round(worst['Macro F1'], 4))
print('Best transfer: ', best.Source, '->', best.Target, round(best['Macro F1'], 4))

src_models, shap_imp = {}, {}
for src in names:
    tr = part(src, 'train'); src_models[src] = fit_xgb(prep_X(tr), tr.y.values, TUNED[f'{src}|XGBoost'], 0)
rows_rq3 = []
for src in names:
    expl = shap.TreeExplainer(src_models[src])
    for tgt in names:
        if src == tgt: continue
        ev = part(tgt, 'ttest').sample(min(2000, len(part(tgt, 'ttest'))), random_state=0)
        sv = expl.shap_values(prep_X(ev))
        imp = pd.Series(np.abs(sv).mean(0), index=FEATURES); shap_imp[(src, tgt)] = imp
        w = imp / imp.sum()
        rows_rq3.append({'Source': src, 'Target': tgt,
                         'Transfer macro F1': float(xp[(xp.Source == src) & (xp.Target == tgt)]['Macro F1'].iloc[0]),
                         'SHAP-weighted KS shift': float((w * shift_pairs[(src, tgt)]).sum()),
                         'Unweighted mean KS shift': float(shift_pairs[(src, tgt)].mean()),
                         'Top-5 SHAP features': ', '.join(imp.sort_values(ascending=False).index[:5])})
rq3 = pd.DataFrame(rows_rq3)
rho_w, p_w = spearmanr(rq3['SHAP-weighted KS shift'], rq3['Transfer macro F1'])
rho_u, p_u = spearmanr(rq3['Unweighted mean KS shift'], rq3['Transfer macro F1'])
print(f'Spearman (SHAP-weighted shift vs F1): rho={rho_w:.3f}, p={p_w:.4f}')
print(f'Spearman (unweighted shift vs F1):   rho={rho_u:.3f}, p={p_u:.4f}')
rq3.loc[len(rq3)] = {'Source': 'Spearman rho (p)', 'Target': '',
                     'Transfer macro F1': np.nan, 'SHAP-weighted KS shift': f'{rho_w:.3f} ({p_w:.4f})',
                     'Unweighted mean KS shift': f'{rho_u:.3f} ({p_u:.4f})', 'Top-5 SHAP features': ''}
save_table(rq3.round(4), 'table5_7_shift_vs_transfer'); display(rq3)

In [ ]:
# Figure 5.8: SHAP importance, best vs worst transfer
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={'wspace': 0.85})
for ax, (lab, r) in zip(axes, [('Best', best), ('Worst', worst)]):
    imp = shap_imp[(r.Source, r.Target)].sort_values(ascending=True).tail(12)
    ks = shift_pairs[(r.Source, r.Target)][imp.index]
    ax.barh(imp.index, imp.values, color=plt.cm.magma_r(ks.values))
    ax.set_title(f'{lab} transfer: {r.Source} \u2192 {r.Target} (F1 = {r["Macro F1"]:.2f})', fontsize=10)
    ax.set_xlabel('Mean |SHAP value|')
sm = plt.cm.ScalarMappable(cmap='magma_r', norm=plt.Normalize(0, 1)); fig.colorbar(sm, ax=axes, label='KS shift between source and target')
save_fig(fig, 'fig5_8_shap_best_worst'); plt.show()

# Figure 5.9: shift vs transfer performance
fig, ax = plt.subplots(figsize=(5.5, 4))
d = rq3.dropna(subset=['Transfer macro F1'])
ax.scatter(d['SHAP-weighted KS shift'].astype(float), d['Transfer macro F1'], s=50)
for _, r in d.iterrows(): ax.annotate(f'{r.Source[:4]}\u2192{r.Target[:4]}', (float(r['SHAP-weighted KS shift']), r['Transfer macro F1']), fontsize=7)
ax.set_xlabel('SHAP-weighted feature shift (KS)'); ax.set_ylabel('Transfer macro F1 (XGBoost)')
ax.set_title(f'Spearman rho = {rho_w:.2f} (p = {p_w:.3f})')
save_fig(fig, 'fig5_9_shift_vs_f1'); plt.show()

# Figure 5.10: distributions of the top-3 SHAP features in the worst transfer
top3 = shap_imp[(worst.Source, worst.Target)].sort_values(ascending=False).index[:3]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, f in zip(axes, top3):
    for nm, col in [(worst.Source, 'tab:blue'), (worst.Target, 'tab:red')]:
        v = prep_X(data[nm].sample(min(20000, len(data[nm])), random_state=0), [f]).ravel()
        ax.hist(v, bins=50, alpha=0.5, density=True, color=col, label=nm)
    ax.set_title(f'{f}\nKS = {shift_pairs[(worst.Source, worst.Target)][f]:.2f}', fontsize=9); ax.set_xlabel('sign(x)·log(1+|x|)')
axes[0].legend(fontsize=8)
save_fig(fig, 'fig5_10_worst_transfer_features'); plt.show()

In [ ]:
# Predefined feature removals, with equal-size random controls (XGBoost).
keys,ablation_rows=cache_load('ablations')
for tgt in data:
    trn=source_union(tgt,'train',cap=MAX_TRAIN_ROWS);ev=part(tgt,'ttest')
    groups={'TTL':['MIN_TTL','MAX_TTL'],'Window':['TCP_WIN_MAX_IN','TCP_WIN_MAX_OUT'],
            'TTL_Window':['MIN_TTL','MAX_TTL','TCP_WIN_MAX_IN','TCP_WIN_MAX_OUT']}
    for group,drop in groups.items():
        for seed in SEEDS:
            for control in ['specified','random']:
                key=f'{tgt}|{group}|{seed}|{control}'
                if key in keys:continue
                removed=drop if control=='specified' else list(np.random.default_rng(seed+71).choice(FEATURES,len(drop),replace=False))
                feats=[f for f in FEATURES if f not in removed]
                model=fit_xgb(prep_X(trn,feats),trn.y.values,TUNED[f'LODO-{tgt}|XGBoost'],seed)
                row={'Target':tgt,'Group':group,'Control':control,'Seed':seed,'Removed':removed,
                     **metrics(ev.y.values,score_xgb(model,prep_X(ev,feats)))}
                cache_add('ablations',key,[row]);ablation_rows.append(row)
save_table(pd.DataFrame(ablation_rows),'feature_ablation_runs')
# Trivial baselines on exactly the same target test set.
baselines=[]
for tgt in data:
    ev=part(tgt,'ttest')
    for name,sc in [('All benign',np.zeros(len(ev))),('All attack',np.ones(len(ev))),
                    ('Random 50%',np.random.default_rng(0).random(len(ev)))]:
        baselines.append({'Target':tgt,'Baseline':name,**metrics(ev.y.values,sc)})
save_table(pd.DataFrame(baselines),'trivial_baselines')


## Part 7: Export
Saves the S2 feature lists and tuned hyperparameters (Table 4.3), then zips everything. Download `thesis_results.zip` from Drive and upload the ZIP and completed notebook for preparation of the full thesis in its original format.

In [ ]:
hp = pd.DataFrame([{'Setting': k.split('|')[0], 'Model': k.split('|')[1], 'Selected hyperparameters': json.dumps(v, default=str)} for k, v in TUNED.items()])
save_table(hp, 'table4_3_selected_hyperparameters')
save_table(pd.DataFrame([{'Target (held out)': t, 'S2 features': ', '.join(f)} for t, f in S2_FEATS.items()]), 'table5_8_s2_features')
zpath = f'{RESULTS_DIR}/thesis_results.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{RESULTS_DIR}/tables/*') + glob.glob(f'{RESULTS_DIR}/figures/*') + glob.glob(f'{RESULTS_DIR}/logs/*') + glob.glob(f'{RESULTS_DIR}/checkpoints/*.json'):
        z.write(f, os.path.relpath(f, RESULTS_DIR))
print('Done. Results zip:', zpath, '| QUICK_MODE =', QUICK_MODE)
print('UPLOAD THIS ZIP to ChatGPT:',zpath)
print('Also save the notebook with outputs. Do not use QUICK_MODE results in the thesis.')
from google.colab import files
files.download(zpath)
